In [35]:
import os
from pathlib import Path

import polars as pl
import seaborn as sns
from sklearn.model_selection import cross_val_score
from sklearn.ensemble import RandomForestClassifier
import numpy as np

from simple_evals.improvement.models.results import AllResults
from simple_evals.improvement.models.rag_logs import RAGLog
from simple_evals.improvement.paths import RESULTS_DIR

Columns we need:
- prompt_id
- similarity_score
- theme
- effect_of_rag


In [11]:
train_test = pl.read_csv("train_test.csv")

In [3]:
baseline_results_path = RESULTS_DIR / (
    "5df4ba309cb03369f6663786ae6a9904385524a9/"
    + "maverick/healthbench_llama-4-maverick_20251023_212754_allresults.json"
)
baseline_results = AllResults.from_file(baseline_results_path)

In [4]:
rag_run_dir = (
    RESULTS_DIR / "cbd99b81af7e1cb59d122dec8d0cb78717b8d10d/llama-4-maverick-rag2"
)
rag_results_path = (
    rag_run_dir / "healthbench_llama-4-maverick-rag2_20251120_175439_allresults.json"
)
rag_logs_dir = rag_run_dir / "rag_info"
rag_results = AllResults.from_file(rag_results_path)
rag_logs = RAGLog.from_log_dir(rag_logs_dir)

In [5]:
def flatten_metadata(results: AllResults, model_name: str) -> list[dict]:
    """
    Flattens the 'example_level_metadata' objects from the benchmark results file.
    """
    out = []
    for metadata in results.metadata.example_level_metadata:
        out.append(
            {
                "prompt_id": metadata.prompt_id,
                f"score_{model_name}": metadata.score,
            }
        )
    return out


def flatten_rag_logs(rag_logs: list[RAGLog]) -> list[dict]:
    """
    Flattens the rag_logs into dictionaries. It does this by leaving off the
    "conversations" key.
    """
    out = []
    for log in rag_logs:
        out.append(
            {
                "prompt_id": log.prompt_id,
                "vector_search_row_id": log.vector_search_row_id,
                "atropos_case_id": log.atropos_case_id,
                "similarity_score": log.similarity_score,
            }
        )
    return out

In [6]:
baseline = pl.DataFrame(flatten_metadata(baseline_results, "baseline"))
rag = pl.DataFrame(flatten_metadata(rag_results, "rag"))
rag_logs_df = pl.DataFrame(flatten_rag_logs(rag_logs))

In [12]:
combined = (
    baseline.join(rag, on="prompt_id")
    .join(rag_logs_df, on="prompt_id")
    .join(train_test, on="prompt_id")
)
combined.head()

prompt_id,score_baseline,score_rag,vector_search_row_id,atropos_case_id,similarity_score,theme,train_test
str,f64,f64,f64,str,f64,str,str
"""1f548d5b-cd00-49a0-b327-283a2e…",0.637363,0.483516,8431.0,"""ea4beae57e894b2da157caa657cc5b…",0.003307,"""context_seeking""","""train"""
"""0b8f1d60-2081-4562-98f7-b6a976…",0.338983,0.237288,1401.0,"""25206c98a3ca4ce594365ff1664ecc…",0.002424,"""communication""","""test"""
"""6f7a2ee9-e9c6-42d8-b79f-22dea9…",1.0,1.0,8551.0,"""dc800050b3f04b1bb19a94aff5da21…",0.002118,"""emergency_referrals""","""test"""
"""19ec4833-86e9-4166-8b82-d1da09…",0.469136,0.37037,39.0,"""c86238f20fed43c9bea953bd2c2b19…",0.002118,"""hedging""","""test"""
"""7ebc830a-8dbd-489b-9d61-4d8bac…",0.257143,0.1,3925.0,"""11a057dc3ae4482094b98653e1b052…",0.001943,"""emergency_referrals""","""test"""


In [15]:
diffs = combined.with_columns(
    # Calculate the difference between the runs for each question
    (pl.col("score_rag") - pl.col("score_baseline")).alias("effect_of_rag"),
    # Calculate the difference between the runs for each question
    (
        pl.when((pl.col("score_rag") - pl.col("score_baseline") > 0))
        .then(pl.lit("better"))
        .otherwise(pl.lit("worse"))
    ).alias("better_or_worse"),
).filter(pl.col("train_test") == "train")

In [16]:
diffs

prompt_id,score_baseline,score_rag,vector_search_row_id,atropos_case_id,similarity_score,theme,train_test,effect_of_rag,better_or_worse
str,f64,f64,f64,str,f64,str,str,f64,str
"""1f548d5b-cd00-49a0-b327-283a2e…",0.637363,0.483516,8431.0,"""ea4beae57e894b2da157caa657cc5b…",0.003307,"""context_seeking""","""train""",-0.153846,"""worse"""
"""c971f9d1-5f6a-464e-b282-41c8f0…",0.245614,0.0,906.0,"""1f913b0e241744ef97977b32db1620…",0.002817,"""global_health""","""train""",-0.245614,"""worse"""
"""5a6e4a41-3ea6-4050-a971-93433f…",0.086957,0.130435,4635.0,"""2ab5b5d56fee4c15be2906ad92523e…",0.002146,"""global_health""","""train""",0.043478,"""better"""
"""8f2a65de-dea7-48e8-8adb-6194ec…",-0.119403,-0.119403,7847.0,"""ecf87fb6a6244c78910b5a4577b265…",0.001861,"""global_health""","""train""",0.0,"""worse"""
"""6d5f483c-3e86-456d-bfd5-4e28de…",0.048276,0.213793,308.0,"""dc3e5f01460346ebae5ca141a88eda…",0.002251,"""global_health""","""train""",0.165517,"""better"""
…,…,…,…,…,…,…,…,…,…
"""6c1eaabb-908a-4414-8d73-597b53…",1.0,0.558824,435.0,"""1ccdbde4a27346b7b1465497764a73…",0.001859,"""health_data_tasks""","""train""",-0.441176,"""worse"""
"""c4d8b028-f148-47ca-b1bc-c5c467…",-0.692308,0.461538,10755.0,"""1f7a7a4f4dc34f5fa96a027a16e0fc…",0.002552,"""health_data_tasks""","""train""",1.153846,"""better"""
"""0175eaea-4d14-4932-956b-601951…",0.388889,-0.111111,7965.0,"""09b63f655a9a4995b2c164b116bf7e…",0.003114,"""communication""","""train""",-0.5,"""worse"""


In [ ]:
# impact code the themes
impact_codes = diffs.group_by("theme").agg(
    pl.col("effect_of_rag").mean().alias("theme_impact_coded")
)
impact_codes

theme,theme_impact_coded
str,f64
"""global_health""",-0.06562
"""complex_responses""",-0.011168
"""emergency_referrals""",-0.068764
"""health_data_tasks""",-0.00074
"""context_seeking""",-0.028737
"""hedging""",-0.052157
"""communication""",-0.021714


In [23]:
train_df = diffs.join(impact_codes, on="theme").with_columns(
    pl.when(pl.col("better_or_worse") == "better")
    .then(pl.lit(1))
    .otherwise(pl.lit(0))
    .alias("better_or_worse_binary")
)
train_df

prompt_id,score_baseline,score_rag,vector_search_row_id,atropos_case_id,similarity_score,theme,train_test,effect_of_rag,better_or_worse,theme_impact_coded,better_or_worse_binary
str,f64,f64,f64,str,f64,str,str,f64,str,f64,i32
"""1f548d5b-cd00-49a0-b327-283a2e…",0.637363,0.483516,8431.0,"""ea4beae57e894b2da157caa657cc5b…",0.003307,"""context_seeking""","""train""",-0.153846,"""worse""",-0.028737,0
"""c971f9d1-5f6a-464e-b282-41c8f0…",0.245614,0.0,906.0,"""1f913b0e241744ef97977b32db1620…",0.002817,"""global_health""","""train""",-0.245614,"""worse""",-0.06562,0
"""5a6e4a41-3ea6-4050-a971-93433f…",0.086957,0.130435,4635.0,"""2ab5b5d56fee4c15be2906ad92523e…",0.002146,"""global_health""","""train""",0.043478,"""better""",-0.06562,1
"""8f2a65de-dea7-48e8-8adb-6194ec…",-0.119403,-0.119403,7847.0,"""ecf87fb6a6244c78910b5a4577b265…",0.001861,"""global_health""","""train""",0.0,"""worse""",-0.06562,0
"""6d5f483c-3e86-456d-bfd5-4e28de…",0.048276,0.213793,308.0,"""dc3e5f01460346ebae5ca141a88eda…",0.002251,"""global_health""","""train""",0.165517,"""better""",-0.06562,1
…,…,…,…,…,…,…,…,…,…,…,…
"""6c1eaabb-908a-4414-8d73-597b53…",1.0,0.558824,435.0,"""1ccdbde4a27346b7b1465497764a73…",0.001859,"""health_data_tasks""","""train""",-0.441176,"""worse""",-0.00074,0
"""c4d8b028-f148-47ca-b1bc-c5c467…",-0.692308,0.461538,10755.0,"""1f7a7a4f4dc34f5fa96a027a16e0fc…",0.002552,"""health_data_tasks""","""train""",1.153846,"""better""",-0.00074,1
"""0175eaea-4d14-4932-956b-601951…",0.388889,-0.111111,7965.0,"""09b63f655a9a4995b2c164b116bf7e…",0.003114,"""communication""","""train""",-0.5,"""worse""",-0.021714,0


In [ ]:
train_df.select(
    "similarity_score", "theme_impact_coded", "better_or_worse_binary"
).describe()

statistic,similarity_score,theme_impact_coded,better_or_worse_binary
str,f64,f64,f64
"""count""",2500.0,2500.0,2500.0
"""null_count""",0.0,0.0,0.0
"""mean""",0.002181,-0.040497,0.3136
"""std""",0.000441,0.023208,0.464049
"""min""",0.001379,-0.068764,0.0
"""25%""",0.001861,-0.06562,0.0
"""50%""",0.002088,-0.052157,0.0
"""75%""",0.002406,-0.021714,1.0
"""max""",0.004808,-0.00074,1.0


In [ ]:
model = RandomForestClassifier(random_state=42)
aucs = cross_val_score(
    model,
    X=train_df.select("similarity_score", "theme_impact_coded").to_numpy(),
    y=train_df["better_or_worse_binary"].to_numpy(),
    cv=10,
    scoring="roc_auc",
)

In [ ]:
aucs

array([0.56436345, 0.55403995, 0.51229875, 0.52817531, 0.51073345,
       0.45087955, 0.50862388, 0.48201199, 0.47494263, 0.50281294])

In [ ]:
np.mean(aucs)

np.float64(0.5088881901895126)